In [4]:
import pandas as pd
import numpy as np

# Charger les données
data = pd.read_csv('uber.csv')

print("ÉTAPE 1 : Chargement et exploration")


print(f"\n Fichier chargé!")
print(f"  Dimensions: {data.shape[0]} lignes, {data.shape[1]} colonnes")

print(f"\n Premières lignes:")
print(data.head())

print(f"\n Infos sur le dataset:")
print(data.info())

print(f"\n Valeurs manquantes:")
print(data.isnull().sum())

print(f"\n Statistiques:")
print(data.describe())

print(f"\n Colonnes disponibles:")
print(data.columns.tolist())

ÉTAPE 1 : Chargement et exploration

 Fichier chargé!
  Dimensions: 200000 lignes, 9 colonnes

 Premières lignes:
   Unnamed: 0                            key  fare_amount  \
0    24238194    2015-05-07 19:52:06.0000003          7.5   
1    27835199    2009-07-17 20:04:56.0000002          7.7   
2    44984355   2009-08-24 21:45:00.00000061         12.9   
3    25894730    2009-06-26 08:22:21.0000001          5.3   
4    17610152  2014-08-28 17:47:00.000000188         16.0   

           pickup_datetime  pickup_longitude  pickup_latitude  \
0  2015-05-07 19:52:06 UTC        -73.999817        40.738354   
1  2009-07-17 20:04:56 UTC        -73.994355        40.728225   
2  2009-08-24 21:45:00 UTC        -74.005043        40.740770   
3  2009-06-26 08:22:21 UTC        -73.976124        40.790844   
4  2014-08-28 17:47:00 UTC        -73.925023        40.744085   

   dropoff_longitude  dropoff_latitude  passenger_count  
0         -73.999512         40.723217                1  
1         -7

In [5]:
print("ÉTAPE 2 : Nettoyage et préparation")


#  Supprimer les lignes avec valeurs manquantes
print("\n Suppression des valeurs manquantes...")
print(f"   Avant: {len(data)} lignes")
data = data.dropna()
print(f"   Après: {len(data)} lignes")

#  Supprimer les tarifs aberrants
print("\n Suppression des tarifs aberrants...")
print(f"   Avant: {len(data)} lignes")
print(f"   Tarif min: ${data['fare_amount'].min():.2f}")
print(f"   Tarif max: ${data['fare_amount'].max():.2f}")

# Garder entre 2.5 et 200 dollars
data = data[(data['fare_amount'] > 2.5) & (data['fare_amount'] < 200)]
print(f"   Après: {len(data)} lignes")
print(f"   Nouveau tarif min: ${data['fare_amount'].min():.2f}")
print(f"   Nouveau tarif max: ${data['fare_amount'].max():.2f}")

#  Supprimer les coordonnées aberrantes
print("\n Suppression des coordonnées aberrantes...")
print(f"   Avant: {len(data)} lignes")

# New York: longitude entre -74.02 et -73.93, latitude entre 40.71 et 40.80
data = data[
    (data['pickup_longitude'] >= -74.02) & (data['pickup_longitude'] <= -73.93) &
    (data['pickup_latitude'] >= 40.71) & (data['pickup_latitude'] <= 40.80) &
    (data['dropoff_longitude'] >= -74.02) & (data['dropoff_longitude'] <= -73.93) &
    (data['dropoff_latitude'] >= 40.71) & (data['dropoff_latitude'] <= 40.80)
]
print(f"   Après: {len(data)} lignes")

#  Convertir en datetime
print("\n Conversion en datetime...")
data['pickup_datetime'] = pd.to_datetime(data['pickup_datetime'])
print(f"    Conversion réussie")

#  Extraire les features temporelles
print("\n Extraction des features temporelles...")
data['hour'] = data['pickup_datetime'].dt.hour
data['day_of_week'] = data['pickup_datetime'].dt.dayofweek  # 0=Lundi, 6=Dimanche
data['month'] = data['pickup_datetime'].dt.month
print(f"    Nouvelles colonnes créées: hour, day_of_week, month")

#  Calculer la distance
print("\n Calcul de la distance...")
data['distance'] = np.sqrt(
    (data['dropoff_longitude'] - data['pickup_longitude'])**2 * (111**2) +
    (data['dropoff_latitude'] - data['pickup_latitude'])**2 * (111**2)
)

# Supprimer les trajets avec distance = 0
data = data[data['distance'] > 0.1]
print(f"   Distance moyenne: {data['distance'].mean():.2f} km")
print(f"   Distance min: {data['distance'].min():.2f} km")
print(f"   Distance max: {data['distance'].max():.2f} km")

print(f"\n Nettoyage terminé!")
print(f"  Lignes restantes: {len(data)}")

print(f"\nAperçu des données nettoyées:")
print(data[['fare_amount', 'distance', 'hour', 'day_of_week']].head(10))

ÉTAPE 2 : Nettoyage et préparation

 Suppression des valeurs manquantes...
   Avant: 200000 lignes
   Après: 199999 lignes

 Suppression des tarifs aberrants...
   Avant: 199999 lignes
   Tarif min: $-52.00
   Tarif max: $499.00
   Après: 199141 lignes
   Nouveau tarif min: $2.60
   Nouveau tarif max: $196.00

 Suppression des coordonnées aberrantes...
   Avant: 199141 lignes
   Après: 155263 lignes

 Conversion en datetime...
    Conversion réussie

 Extraction des features temporelles...
    Nouvelles colonnes créées: hour, day_of_week, month

 Calcul de la distance...
   Distance moyenne: 2.53 km
   Distance min: 0.10 km
   Distance max: 13.10 km

 Nettoyage terminé!
  Lignes restantes: 153425

Aperçu des données nettoyées:
    fare_amount  distance  hour  day_of_week
0           7.5  1.680517    19            3
1           7.7  2.453416    20            4
2          12.9  5.895055    21            0
8           9.7  3.066171     9            4
9          12.5  5.605007    19       

In [7]:
from sklearn.model_selection import train_test_split

print("ÉTAPE 3 : Séparation des données (Train/Test)")


# Sélectionner les features et la cible
X = data[['distance', 'hour', 'day_of_week']]
y = data['fare_amount']

print(f"\n Informations:")
print(f"  Features (X): {X.columns.tolist()}")
print(f"  Cible (y): fare_amount")
print(f"  Nombre total d'exemples: {len(X)}")

# Vérifier s'il y a des NaN
print(f"\n Vérification des NaN:")
print(f"  NaN dans X: {X.isnull().sum().sum()}")
print(f"  NaN dans y: {y.isnull().sum()}")

# Diviser en train (80%) et test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42
)

print(f"\n Données divisées:")
print(f"  Ensemble d'entraînement (80%): {len(X_train)} exemples")
print(f"  Ensemble de test (20%): {len(X_test)} exemples")

print(f"\n Statistiques de y_train:")
print(y_train.describe())

print(f"\n Statistiques de y_test:")
print(y_test.describe())

ÉTAPE 3 : Séparation des données (Train/Test)

 Informations:
  Features (X): ['distance', 'hour', 'day_of_week']
  Cible (y): fare_amount
  Nombre total d'exemples: 153425

 Vérification des NaN:
  NaN dans X: 0
  NaN dans y: 0

 Données divisées:
  Ensemble d'entraînement (80%): 122740 exemples
  Ensemble de test (20%): 30685 exemples

 Statistiques de y_train:
count    122740.000000
mean          8.621478
std           4.205408
min           2.900000
25%           5.700000
50%           7.700000
75%          10.500000
max         113.660000
Name: fare_amount, dtype: float64

 Statistiques de y_test:
count    30685.000000
mean         8.607378
std          4.287502
min          2.900000
25%          5.700000
50%          7.700000
75%         10.500000
max        120.300000
Name: fare_amount, dtype: float64


In [8]:
from sklearn.linear_model import LinearRegression

print("ÉTAPE 4 : Entraînement - Régression Linéaire")


# Créer le modèle
lr = LinearRegression()

# Entraîner le modèle
print("\n Entraînement du modèle...")
lr.fit(X_train, y_train)
print(" Modèle entraîné avec succès!")

# Faire des prédictions
y_pred_lr = lr.predict(X_test)
print(f" Prédictions effectuées sur {len(X_test)} exemples")

# Afficher les coefficients
print(f"\n Coefficients du modèle:")
print(f"  Intercept (constante): ${lr.intercept_:.4f}")

ÉTAPE 4 : Entraînement - Régression Linéaire

 Entraînement du modèle...
 Modèle entraîné avec succès!
 Prédictions effectuées sur 30685 exemples

 Coefficients du modèle:
  Intercept (constante): $3.6443


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("ÉTAPE 5 : Évaluation - Régression Linéaire")

# Calculer les métriques
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print(f"\n Métriques d'évaluation:")
print(f"  MAE (Erreur Absolue Moyenne):  ${mae_lr:.2f}")
print(f"     En moyenne, le modèle se trompe de ${mae_lr:.2f}")
print(f"\n  RMSE (Racine Erreur Quadratique): ${rmse_lr:.2f}")
print(f"     Pénalise plus les grosses erreurs")
print(f"\n  R² Score: {r2_lr:.4f}")

# Évaluation de la performance
print(f"\n Évaluation:")
if r2_lr > 0.7:
    print(f"   BON modèle (R² > 0.7)")
elif r2_lr > 0.5:
    print(f"   ACCEPTABLE (0.5 < R² < 0.7)")
elif r2_lr > 0.3:
    print(f"    FAIBLE (0.3 < R² < 0.5)")
else:
    print(f"   TRÈS FAIBLE (R² < 0.3)")

ÉTAPE 5 : Évaluation - Régression Linéaire

 Métriques d'évaluation:
  MAE (Erreur Absolue Moyenne):  $1.71
     En moyenne, le modèle se trompe de $1.71

  RMSE (Racine Erreur Quadratique): $2.91
     Pénalise plus les grosses erreurs

  R² Score: 0.5381

 Évaluation:
   ACCEPTABLE (0.5 < R² < 0.7)


In [13]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('MAE  =', mean_absolute_error(y_test, y_pred_rf))
print('RMSE =', np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print('R²   =', r2_score(y_test, y_pred_rf))

ImportError: DLL load failed while importing _gradient_boosting: An Application Control policy has blocked this file.

In [1]:
import matplotlib.pyplot as plt

plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.xlabel('Valeurs réelles')
plt.ylabel('Valeurs prédites')
plt.title('Prédiction du tarif Uber')
plt.show()

NameError: name 'y_test' is not defined